In [22]:
import pandas as pd 
import time

In [23]:
def fetch_football_data(seasons, leagues):
    all_data = []
    
    for season in seasons:
        for league in leagues:
            url = f"https://www.football-data.co.uk/mmz4281/{season}/{league}.csv"
            
            try:
                df =pd.read_csv(url)
                df['Season'] = season   
                df['League'] = league
                all_data.append(df)
                
            except Exception as e:
                print(f"Error: {league} {season}: {e}") 
    final_df = pd.concat(all_data, ignore_index=True)
    return final_df


In [24]:
seasons = ['2122', '2223', '2324', '2425', '2526']
leagues = ['E0', 'SP1', 'I1', 'D1', 'F1']

big_df = fetch_football_data(seasons, leagues)


print(f"downloaded matches: {len(big_df)} ")
big_df.to_csv('../data/top5_leagues_raw_combined.csv', index=False)

C:\Users\huber\AppData\Local\Temp\ipykernel_12812\2147237358.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Season'] = season
C:\Users\huber\AppData\Local\Temp\ipykernel_12812\2147237358.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['League'] = league


downloaded matches: 8447 


In [25]:
big_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8447 entries, 0 to 8446
Columns: 164 entries, Div to LBCA
dtypes: float64(152), int64(2), str(10)
memory usage: 10.6 MB


In [26]:
big_df.isnull().sum().sort_values(ascending=False).head(20)

LBCA      7423
LBCD      7423
CLCA      7423
LBCH      7423
CLCH      7423
CLCD      7423
CLA       7402
CLD       7402
LBD       7402
LBH       7402
LBA       7402
CLH       7402
BMGMCA    7156
BVCH      7156
BFDCH     7156
BVA       7156
BFDH      7156
BMGMH     7156
BFDCD     7156
BFDCA     7156
dtype: int64

In [27]:
my_features = [
    'Date', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'FTR', 
    'HTHG', 'HTAG', 'HTR',
    'HS', 'AS', 'HST', 'AST', 
    'HR', 'AR', 
    'B365H', 'B365D', 'B365A'
    'Season', 'League'
]

In [28]:
big_df_clean = big_df[my_features].copy()

KeyError: "['B365ASeason'] not in index"

In [ ]:
missing_values_clean = big_df_clean.isnull().mean() *100
print("percentage of missing values in cleaned dataset:")
print(missing_values_clean)

percentage of missing values in cleaned dataset:
Date        0.000000
HomeTeam    0.000000
AwayTeam    0.000000
FTHG        0.000000
FTAG        0.000000
FTR         0.000000
HTHG        0.011839
HTAG        0.011839
HTR         0.011839
HS          0.011839
AS          0.011839
HST         0.011839
AST         0.011839
HR          0.011839
AR          0.011839
B365H       0.011839
B365D       0.011839
B365A       0.011839
dtype: float64


In [ ]:
big_df_clean = big_df_clean.dropna()

In [ ]:
rename_mapping = {
    'HomeTeam': 'home_team',
    'AwayTeam': 'away_team',
    'FTHG': 'home_goals',
    'FTAG': 'away_goals',
    'FTR': 'match_result',
    'HS': 'home_shots',
    'AS': 'away_shots',
    'HST': 'home_shots_target',
    'AST': 'away_shots_target',
    'HR': 'home_red_cards',
    'AR': 'away_red_cards',
    'HTHG': 'ht_home_goals',
    'HTAG': 'ht_away_goals',
    'HTR': 'ht_result'
}

big_df_clean.rename(columns=rename_mapping, inplace=True)

In [ ]:
big_df_clean['Date'] = pd.to_datetime(big_df_clean['Date'])

C:\Users\huber\AppData\Local\Temp\ipykernel_12812\3592665718.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  big_df_clean['Date'] = pd.to_datetime(big_df_clean['Date'])


In [ ]:
big_df_clean = big_df_clean.sort_values('Date')
big_df_clean = big_df_clean.reset_index(drop=True)

In [ ]:
df['home_team_goals_avg_last_5'] = df.groupby('home_team')['home_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['away_goals_avg_last_5'] = df.groupby('away_team')['away_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['ht_home_goals_avg_last_5'] = df.groupby('home_team')['ht_home_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['ht_away_goals_avg_last_5'] = df.groupby('away_team')['ht_away_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [ ]:
df['home_balance_1h'] = df['ht_home_goals'] - df['ht_away_goals']
df['home_balance_2h'] = (df['home_goals'] - df['ht_home_goals']) - (df['away_goals'] - df['ht_away_goals'])
df['away_balance_1h'] = -df['home_balance_1h']
df['away_balance_2h'] = -df['home_balance_2h']

In [ ]:
df['home_form_1h_last_5'] = df.groupby('home_team')['home_balance_1h'].transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
df['away_form_1h_last_5'] = df.groupby('away_team')['away_balance_1h'].transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
df['home_form_2h_last_5'] = df.groupby('home_team')['home_balance_2h'].transform( lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['away_form_2h_last_5'] = df.groupby('away_team')['away_balance_2h'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [ ]:
df['home_shots_avg_last_5'] = df.groupby('home_team')['home_shots'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['away_shots_avg_last_5'] = df.groupby('away_team')['away_shots'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [ ]:
df['home_shots_target_avg_last_5'] = df.groupby('home_team')['home_shots_target'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['away_shots_target_avg_last_5'] = df.groupby('away_team')['away_shots_target'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [ ]:
df['home_red_cards_avg_last_5'] = df.groupby('home_team')['home_red_cards'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['away_red_cards_avg_last_5'] = df.groupby('away_team')['away_red_cards'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [ ]:
df['home_goals_conceded_avg_last_5'] = df.groupby('home_team')['away_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['away_goals_conceded_avg_last_5'] = df.groupby('away_team')['home_goals'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [ ]:
df['target'] = df['match_result'].map({'H': 2, 'D': 1, 'A': 0})

In [ ]:
df['home_points'] = df['match_result'].map({'H': 3, 'D': 1, 'A': 0})
df['away_points'] = df['match_result'].map({'H': 0, 'D': 1, 'A': 3})

In [ ]:
df['home_points_avg_last_5'] = df.groupby('home_team')['home_points'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
df['away_points_avg_last_5'] = df.groupby('away_team')['away_points'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [ ]:
df_home_points = df[['Date', 'home_team', 'home_points']].copy()
df_away_points = df[['Date', 'away_team', 'away_points']].copy()
df_home_points.columns = ['Date', 'Team', 'Points']
df_away_points.columns = ['Date', 'Team', 'Points']
concatenated_points = pd.concat([df_home_points, df_away_points], ignore_index=True)
concated_points_sorted = concatenated_points.sort_values(by=['Team', 'Date'])
concated_points_sorted['overall_points_last_5'] = concated_points_sorted.groupby('Team')['Points'].transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))

In [ ]:
df = df.merge(
    concated_points_sorted[['Date', 'Team', 'overall_points_last_5']],
    left_on=['Date', 'home_team'],   
    right_on=['Date', 'Team'],       
    how='left'
)
df = df.drop(columns=['Team'])
df = df.rename(columns={'overall_points_last_5': 'home_overall_points_last_5'})

df = df.merge(
    concated_points_sorted[['Date', 'Team', 'overall_points_last_5']],
    left_on=['Date', 'away_team'],
    right_on=['Date', 'Team'],
    how='left'
)
df = df.drop(columns=['Team'])
df = df.rename(columns={'overall_points_last_5': 'away_overall_points_last_5'})